# Notebook 05: RAG Indexing

## Objetivo
Crear indices duales (FAISS + BM25) para retrieval hibrido.

```
rag_docs.jsonl      → Indice FAISS (pseudo_label)    + Indice BM25 (keywords)
rag_evidence.jsonl  → Indice FAISS (raw_evidence)    + Indice BM25 (texto bruto)
```

## Estrategia de retrieval
| Metodo | Captura | Debilidad |
|--------|---------|----------|
| **Vector (FAISS)** | Similitud semantica | No captura terminos exactos |
| **Lexico (BM25)** | Terminos exactos (PCA, BERT) | No entiende sinonimos |
| **Hibrido** | Lo mejor de ambos | |

## Inputs
- `artifacts/rag/rag_docs.jsonl` — documentos enriquecidos
- `artifacts/rag/rag_evidence.jsonl` — documentos evidencia

## Outputs
- `.rag_index/pseudo_label/` — FAISS index + metadata
- `.rag_index/raw_evidence/` — FAISS index + metadata
- `.rag_index/bm25_pseudo_label.pkl` — BM25 index
- `.rag_index/bm25_raw_evidence.pkl` — BM25 index

---
## Setup

In [ ]:
import json
import pickle
import math
import re
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from collections import Counter
import numpy as np

# Paths — detect Proyecto_Final_Alejandro/notebooks/ and go up 2 levels
cwd = Path.cwd()
if cwd.name == "notebooks" and cwd.parent.name == "Proyecto_Final_Alejandro":
    PROJECT_ROOT = cwd.parent.parent
elif cwd.name in ("notebooks", "Proyecto_Final_Alejandro"):
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd
RAG_DIR = PROJECT_ROOT / "artifacts" / "rag"
INDEX_DIR = PROJECT_ROOT / ".rag_index"
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print(f"RAG data:  {RAG_DIR}")
print(f"Index dir: {INDEX_DIR}")

In [ ]:
# Verificar dependencias
try:
    import faiss
    print(f"FAISS: OK (version {faiss.__version__ if hasattr(faiss, '__version__') else 'installed'})")
except ImportError:
    print("FAISS no instalado. Ejecuta: pip install faiss-cpu")
    raise

try:
    from sentence_transformers import SentenceTransformer
    print(f"SentenceTransformers: OK")
except ImportError:
    print("sentence-transformers no instalado. Ejecuta: pip install sentence-transformers")
    raise

---
## 1. Cargar documentos RAG

In [ ]:
def load_jsonl(path: Path) -> List[Dict]:
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

docs_path = RAG_DIR / "rag_docs.jsonl"
evidence_path = RAG_DIR / "rag_evidence.jsonl"

if not docs_path.exists():
    raise FileNotFoundError(f"Ejecuta primero RAG_01_Data_Preparation.ipynb. No existe: {docs_path}")

rag_docs = load_jsonl(docs_path)
rag_evidence = load_jsonl(evidence_path) if evidence_path.exists() else []

print(f"Documentos enriquecidos: {len(rag_docs)}")
print(f"Documentos evidencia:    {len(rag_evidence)}")

# Quick stats
if rag_docs:
    avg_len = sum(len(d["page_content"]) for d in rag_docs) / len(rag_docs)
    avg_score = sum(d.get("quality_score", 0) for d in rag_docs) / len(rag_docs)
    print(f"\nEnriquecidos:")
    print(f"  Longitud media: {avg_len:.0f} chars")
    print(f"  Quality media:  {avg_score:.3f}")
    
    topics = Counter(d.get("topic", "?") for d in rag_docs)
    print(f"  Topics unicos:  {len(topics)}")
    print(f"  Top 5 topics:   {topics.most_common(5)}")

---
## 2. Embedding Model

Usamos `paraphrase-multilingual-minilm-l12-v2` (384 dims):
- Multilingue (espanol + ingles)
- Rapido en CPU
- Buen balance calidad/velocidad para RAG educativo

In [ ]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-minilm-l12-v2"

print(f"Cargando modelo: {MODEL_NAME}")
t0 = time.time()
model = SentenceTransformer(MODEL_NAME)
embedding_dim = model.get_sentence_embedding_dimension()
print(f"Modelo cargado en {time.time()-t0:.1f}s")
print(f"Dimensiones: {embedding_dim}")

# Sanity check
test_emb = model.encode(["test de embeddings"], convert_to_numpy=True)
print(f"Test embedding shape: {test_emb.shape}")

---
## 3. FAISS Index: Documentos enriquecidos (principal)

In [ ]:
def build_faiss_index(
    documents: List[Dict],
    model: SentenceTransformer,
    collection_name: str,
    index_dir: Path,
) -> Dict:
    """Construye indice FAISS + guarda metadata."""
    if not documents:
        return {"status": "empty", "count": 0}
    
    # Extraer textos
    texts = [d["page_content"] for d in documents]
    
    # Embeber
    print(f"  Embebiendo {len(texts)} documentos...")
    t0 = time.time()
    embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=True, batch_size=32)
    embed_time = time.time() - t0
    print(f"  Embeddings: {embeddings.shape} en {embed_time:.1f}s")
    
    # Normalizar para cosine similarity (usamos IndexFlatIP despues)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0] = 1  # avoid division by zero
    embeddings_normalized = embeddings / norms
    
    # Crear indice FAISS (Inner Product = cosine similarity con vectores normalizados)
    dim = embeddings_normalized.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings_normalized.astype(np.float32))
    
    # Metadata por documento
    metadatas = []
    for i, doc in enumerate(documents):
        metadatas.append({
            "idx": i,
            "doc_id": doc["doc_id"],
            "page_content": doc["page_content"],
            "source_type": doc["source_type"],
            "topic": doc.get("topic", ""),
            "keywords": doc.get("keywords", []),
            "slide_id": doc.get("slide_id", ""),
            "session_id": doc.get("session_id", ""),
            "start_ms": doc.get("start_ms", 0),
            "end_ms": doc.get("end_ms", 0),
            "quality_score": doc.get("quality_score", 0),
            "semantic_risk": doc.get("semantic_risk", "unknown"),
            "linked_doc_id": doc.get("linked_doc_id", ""),
        })
    
    # Guardar
    collection_dir = index_dir / collection_name
    collection_dir.mkdir(parents=True, exist_ok=True)
    
    faiss.write_index(index, str(collection_dir / "index.faiss"))
    with open(collection_dir / "metadata.pkl", "wb") as f:
        pickle.dump(metadatas, f)
    with open(collection_dir / "embeddings.npy", "wb") as f:
        np.save(f, embeddings_normalized)
    
    print(f"  Index guardado en {collection_dir}")
    
    return {
        "status": "indexed",
        "collection": collection_name,
        "count": len(documents),
        "embedding_dim": dim,
        "embed_time_s": round(embed_time, 1),
        "index_type": "IndexFlatIP (cosine)",
    }


# Indexar documentos enriquecidos
print("Indexando documentos enriquecidos (pseudo_label)...")
result_enriched = build_faiss_index(rag_docs, model, "pseudo_label", INDEX_DIR)
print(f"\nResultado: {json.dumps(result_enriched, indent=2)}")

---
## 4. FAISS Index: Documentos evidencia (secundario)

In [ ]:
# Indexar documentos evidencia
print("Indexando documentos evidencia (raw_evidence)...")
result_evidence = build_faiss_index(rag_evidence, model, "raw_evidence", INDEX_DIR)
print(f"\nResultado: {json.dumps(result_evidence, indent=2)}")

---
## 5. BM25 Index (busqueda lexica)

Implementacion BM25 pura en Python (sin dependencias extra).
Captura terminos exactos que el vector search puede perder: PCA, BERT, SVD, etc.

In [ ]:
class BM25Index:
    """BM25 index puro en Python (Okapi BM25)."""
    
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.doc_count = 0
        self.avg_doc_len = 0
        self.doc_lens = []      # len of each doc
        self.doc_freqs = {}     # term -> number of docs containing term
        self.term_freqs = []    # per-doc term frequencies
        self.documents = []     # metadata for each doc
        self.vocab = set()
    
    @staticmethod
    def tokenize(text: str) -> List[str]:
        """Tokenizacion simple: lowercase + split + filtrar cortos."""
        tokens = re.findall(r'\b[a-záéíóúüñ0-9_-]{2,}\b', text.lower())
        return tokens
    
    def index(self, documents: List[Dict], text_field: str = "page_content"):
        """Indexa documentos."""
        self.documents = documents
        self.doc_count = len(documents)
        
        total_len = 0
        for doc in documents:
            tokens = self.tokenize(doc.get(text_field, ""))
            
            # Also include keywords for better matching
            keywords = doc.get("keywords", [])
            if keywords:
                for kw in keywords:
                    tokens.extend(self.tokenize(kw))
            
            freq = Counter(tokens)
            self.term_freqs.append(freq)
            self.doc_lens.append(len(tokens))
            total_len += len(tokens)
            
            for term in set(tokens):
                self.doc_freqs[term] = self.doc_freqs.get(term, 0) + 1
                self.vocab.add(term)
        
        self.avg_doc_len = total_len / self.doc_count if self.doc_count else 1
    
    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """Busca documentos por BM25 score."""
        query_tokens = self.tokenize(query)
        if not query_tokens:
            return []
        
        scores = []
        for i in range(self.doc_count):
            score = 0.0
            doc_len = self.doc_lens[i]
            tf_dict = self.term_freqs[i]
            
            for term in query_tokens:
                if term not in tf_dict:
                    continue
                
                tf = tf_dict[term]
                df = self.doc_freqs.get(term, 0)
                
                # IDF
                idf = math.log((self.doc_count - df + 0.5) / (df + 0.5) + 1)
                
                # TF normalization
                tf_norm = (tf * (self.k1 + 1)) / (
                    tf + self.k1 * (1 - self.b + self.b * doc_len / self.avg_doc_len)
                )
                
                score += idf * tf_norm
            
            if score > 0:
                scores.append((i, score))
        
        # Sort by score
        scores.sort(key=lambda x: x[1], reverse=True)
        
        results = []
        for idx, score in scores[:top_k]:
            doc = self.documents[idx]
            results.append({
                "doc_id": doc["doc_id"],
                "bm25_score": round(score, 4),
                "page_content": doc["page_content"],
                "topic": doc.get("topic", ""),
                "keywords": doc.get("keywords", []),
                "start_ms": doc.get("start_ms", 0),
                "end_ms": doc.get("end_ms", 0),
                "quality_score": doc.get("quality_score", 0),
                "linked_doc_id": doc.get("linked_doc_id", ""),
            })
        
        return results
    
    def save(self, path: Path):
        """Persiste el indice."""
        with open(path, "wb") as f:
            pickle.dump({
                "k1": self.k1,
                "b": self.b,
                "doc_count": self.doc_count,
                "avg_doc_len": self.avg_doc_len,
                "doc_lens": self.doc_lens,
                "doc_freqs": self.doc_freqs,
                "term_freqs": self.term_freqs,
                "documents": self.documents,
                "vocab": self.vocab,
            }, f)
    
    @classmethod
    def load(cls, path: Path) -> 'BM25Index':
        """Carga indice persistido."""
        with open(path, "rb") as f:
            data = pickle.load(f)
        idx = cls(k1=data["k1"], b=data["b"])
        idx.doc_count = data["doc_count"]
        idx.avg_doc_len = data["avg_doc_len"]
        idx.doc_lens = data["doc_lens"]
        idx.doc_freqs = data["doc_freqs"]
        idx.term_freqs = data["term_freqs"]
        idx.documents = data["documents"]
        idx.vocab = data["vocab"]
        return idx


print("BM25Index definido.")

In [ ]:
# BM25 para documentos enriquecidos
print("Construyendo BM25 para pseudo_label...")
bm25_enriched = BM25Index()
bm25_enriched.index(rag_docs)
bm25_enriched.save(INDEX_DIR / "bm25_pseudo_label.pkl")
print(f"  {bm25_enriched.doc_count} docs, {len(bm25_enriched.vocab)} terminos unicos")

# BM25 para evidencia
print("Construyendo BM25 para raw_evidence...")
bm25_evidence = BM25Index()
bm25_evidence.index(rag_evidence)
bm25_evidence.save(INDEX_DIR / "bm25_raw_evidence.pkl")
print(f"  {bm25_evidence.doc_count} docs, {len(bm25_evidence.vocab)} terminos unicos")

---
## 6. Verificacion: test queries

In [ ]:
def search_faiss(
    query: str,
    model: SentenceTransformer,
    collection_name: str,
    index_dir: Path,
    top_k: int = 3,
) -> List[Dict]:
    """Busca en indice FAISS."""
    collection_dir = index_dir / collection_name
    
    index = faiss.read_index(str(collection_dir / "index.faiss"))
    with open(collection_dir / "metadata.pkl", "rb") as f:
        metadatas = pickle.load(f)
    
    # Embeber query
    query_emb = model.encode([query], convert_to_numpy=True)
    query_emb = query_emb / np.linalg.norm(query_emb)  # normalize
    
    # Search
    scores, indices = index.search(query_emb.astype(np.float32), top_k)
    
    results = []
    for i, idx in enumerate(indices[0]):
        if idx == -1:
            continue
        meta = metadatas[int(idx)]
        results.append({
            "doc_id": meta["doc_id"],
            "similarity": float(scores[0][i]),
            "topic": meta.get("topic", ""),
            "page_content": meta["page_content"][:200],
            "start_ms": meta.get("start_ms", 0),
            "end_ms": meta.get("end_ms", 0),
            "quality_score": meta.get("quality_score", 0),
        })
    
    return results


# Test queries
test_queries = [
    "Que es Bag of Words",
    "Como funciona PCA",
    "overfitting en redes neuronales",
    "learning rate",
    "presentacion del proyecto",
]

print("=" * 70)
print("TEST QUERIES")
print("=" * 70)

for query in test_queries:
    print(f"\n--- Query: '{query}' ---")
    
    # FAISS
    faiss_results = search_faiss(query, model, "pseudo_label", INDEX_DIR, top_k=2)
    print(f"  FAISS (semantico):")
    for r in faiss_results:
        print(f"    [{r['similarity']:.3f}] {r['topic']} (q={r['quality_score']:.2f})")
    
    # BM25
    bm25_results = bm25_enriched.search(query, top_k=2)
    print(f"  BM25 (lexico):")
    for r in bm25_results:
        print(f"    [{r['bm25_score']:.3f}] {r['topic']} (q={r['quality_score']:.2f})")
    if not bm25_results:
        print(f"    (sin resultados)")

---
## 7. Estadisticas del indice

In [ ]:
# Stats
index_stats = {
    "pseudo_label": {
        "faiss_count": result_enriched.get("count", 0),
        "faiss_dim": result_enriched.get("embedding_dim", 0),
        "bm25_docs": bm25_enriched.doc_count,
        "bm25_vocab": len(bm25_enriched.vocab),
    },
    "raw_evidence": {
        "faiss_count": result_evidence.get("count", 0),
        "faiss_dim": result_evidence.get("embedding_dim", 0),
        "bm25_docs": bm25_evidence.doc_count,
        "bm25_vocab": len(bm25_evidence.vocab),
    },
    "model": MODEL_NAME,
}

with open(INDEX_DIR / "index_stats.json", "w") as f:
    json.dump(index_stats, f, indent=2)

# Size on disk
print("Archivos generados:")
total_size = 0
for p in sorted(INDEX_DIR.rglob("*")):
    if p.is_file():
        size_kb = p.stat().st_size / 1024
        total_size += size_kb
        rel = p.relative_to(INDEX_DIR)
        print(f"  {str(rel):40s} {size_kb:8.1f} KB")
print(f"  {'TOTAL':40s} {total_size:8.1f} KB")

---
## 8. Visualizacion del espacio de embeddings

Proyectamos los vectores de 384 dimensiones a 2D para inspeccionar la estructura del espacio semantico: clusters por sesion, por topic, y distribucion general.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#FAFAFA",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "figure.dpi": 100,
})

PALETTE = ["#1B2A4A", "#38BDF8", "#10B981", "#F59E0B", "#8B5CF6", "#EC4899", "#EF4444", "#6366F1"]

# Cargar embeddings del indice principal
emb_path = INDEX_DIR / "pseudo_label" / "embeddings.npy"
with open(emb_path, "rb") as f:
    embeddings_all = np.load(f)

# Cargar metadata
with open(INDEX_DIR / "pseudo_label" / "metadata.pkl", "rb") as f:
    meta_all = pickle.load(f)

print(f"Embeddings: {embeddings_all.shape}")
print(f"Metadatas: {len(meta_all)}")

# Extraer labels
session_ids = [m["session_id"] for m in meta_all]
topics = [m["topic"] for m in meta_all]
quality_scores = [m["quality_score"] for m in meta_all]
unique_sessions = sorted(set(session_ids))
session_to_color = {s: PALETTE[i % len(PALETTE)] for i, s in enumerate(unique_sessions)}

In [ ]:
# --- PCA: varianza explicada y proyeccion 2D ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))

# PCA varianza explicada acumulada
pca_full = PCA(n_components=min(50, embeddings_all.shape[1]))
pca_full.fit(embeddings_all)
cumvar = np.cumsum(pca_full.explained_variance_ratio_) * 100

axes[0].plot(range(1, len(cumvar)+1), cumvar, color=PALETTE[0], linewidth=2, marker="o", markersize=3)
axes[0].axhline(y=90, color=PALETTE[6], linestyle="--", alpha=0.6, label="90% varianza")
axes[0].axhline(y=95, color=PALETTE[3], linestyle="--", alpha=0.6, label="95% varianza")

# Encontrar componentes para 90% y 95%
n_90 = np.searchsorted(cumvar, 90) + 1
n_95 = np.searchsorted(cumvar, 95) + 1
axes[0].axvline(x=n_90, color=PALETTE[6], linestyle=":", alpha=0.4)
axes[0].axvline(x=n_95, color=PALETTE[3], linestyle=":", alpha=0.4)
axes[0].text(n_90 + 0.5, 85, f"n={n_90}", color=PALETTE[6], fontsize=10)
axes[0].text(n_95 + 0.5, 80, f"n={n_95}", color=PALETTE[3], fontsize=10)

axes[0].set_xlabel("Componentes principales")
axes[0].set_ylabel("Varianza explicada acumulada (%)")
axes[0].set_title("PCA — Varianza explicada", fontweight="bold")
axes[0].legend()

# PCA 2D coloreado por sesion
pca_2d = PCA(n_components=2)
emb_pca = pca_2d.fit_transform(embeddings_all)

for sid in unique_sessions:
    mask = [s == sid for s in session_ids]
    idxs = np.where(mask)[0]
    axes[1].scatter(emb_pca[idxs, 0], emb_pca[idxs, 1],
                    c=session_to_color[sid], label=sid.replace("llm_", ""),
                    alpha=0.6, s=20, edgecolors="white", linewidth=0.3)

axes[1].set_xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)")
axes[1].set_ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)")
axes[1].set_title("PCA 2D — Coloreado por sesion", fontweight="bold")
axes[1].legend(fontsize=9, loc="best", framealpha=0.8, ncol=2)

plt.tight_layout()
plt.show()

print(f"PCA: {n_90} componentes para 90% varianza, {n_95} para 95%")

In [ ]:
# --- t-SNE: proyeccion 2D (mejor estructura no lineal) ---
print("Calculando t-SNE (puede tardar unos segundos)...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000, learning_rate="auto", init="pca")
emb_tsne = tsne.fit_transform(embeddings_all)
print(f"t-SNE completado. KL divergence: {tsne.kl_divergence_:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# t-SNE coloreado por sesion
for sid in unique_sessions:
    mask = [s == sid for s in session_ids]
    idxs = np.where(mask)[0]
    axes[0].scatter(emb_tsne[idxs, 0], emb_tsne[idxs, 1],
                    c=session_to_color[sid], label=sid.replace("llm_", ""),
                    alpha=0.6, s=20, edgecolors="white", linewidth=0.3)
axes[0].set_xlabel("t-SNE 1")
axes[0].set_ylabel("t-SNE 2")
axes[0].set_title("t-SNE — Por sesion", fontweight="bold")
axes[0].legend(fontsize=8, loc="best", framealpha=0.8, ncol=2)

# t-SNE coloreado por quality score
sc = axes[1].scatter(emb_tsne[:, 0], emb_tsne[:, 1],
                     c=quality_scores, cmap="RdYlGn", vmin=0.4, vmax=1.0,
                     alpha=0.6, s=20, edgecolors="white", linewidth=0.3)
fig.colorbar(sc, ax=axes[1], shrink=0.8, label="Quality Score")
axes[1].set_xlabel("t-SNE 1")
axes[1].set_ylabel("t-SNE 2")
axes[1].set_title("t-SNE — Por quality score", fontweight="bold")

# t-SNE coloreado por topic (top 8 topics + otros)
topic_counter = Counter(topics)
top_topics = [t for t, _ in topic_counter.most_common(8)]
topic_colors = {t: PALETTE[i % len(PALETTE)] for i, t in enumerate(top_topics)}

for topic in top_topics:
    mask = [t == topic for t in topics]
    idxs = np.where(mask)[0]
    label = topic[:25] + "..." if len(topic) > 25 else topic
    axes[2].scatter(emb_tsne[idxs, 0], emb_tsne[idxs, 1],
                    c=topic_colors[topic], label=label,
                    alpha=0.6, s=20, edgecolors="white", linewidth=0.3)

# Resto de topics en gris
other_mask = [t not in top_topics for t in topics]
other_idxs = np.where(other_mask)[0]
if len(other_idxs) > 0:
    axes[2].scatter(emb_tsne[other_idxs, 0], emb_tsne[other_idxs, 1],
                    c="#CCCCCC", label=f"Otros ({len(other_idxs)})",
                    alpha=0.3, s=10)

axes[2].set_xlabel("t-SNE 1")
axes[2].set_ylabel("t-SNE 2")
axes[2].set_title("t-SNE — Por topic (top 8)", fontweight="bold")
axes[2].legend(fontsize=7, loc="best", framealpha=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# --- Distribucion de similitudes coseno + histograma de normas ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribucion de similitudes coseno (sample para eficiencia)
np.random.seed(42)
n_sample = min(500, len(embeddings_all))
sample_idx = np.random.choice(len(embeddings_all), n_sample, replace=False)
sample_emb = embeddings_all[sample_idx]

# Matriz de similitudes (upper triangle)
sim_matrix = sample_emb @ sample_emb.T
upper_tri = sim_matrix[np.triu_indices(n_sample, k=1)]

axes[0].hist(upper_tri, bins=60, color=PALETTE[1], edgecolor="white", linewidth=0.3, alpha=0.85)
axes[0].axvline(np.mean(upper_tri), color=PALETTE[6], linestyle="--", linewidth=2,
                label=f"Media: {np.mean(upper_tri):.3f}")
axes[0].axvline(np.median(upper_tri), color=PALETTE[3], linestyle="--", linewidth=2,
                label=f"Mediana: {np.median(upper_tri):.3f}")
axes[0].set_xlabel("Similitud coseno")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribucion de similitudes entre documentos", fontweight="bold")
axes[0].legend(fontsize=9)

# Heatmap de similitud entre sesiones (media)
session_means = {}
for sid in unique_sessions:
    mask = [s == sid for s in session_ids]
    idxs = np.where(mask)[0]
    if len(idxs) > 0:
        session_means[sid] = embeddings_all[idxs].mean(axis=0)

session_list = sorted(session_means.keys())
n_sess = len(session_list)
sim_sessions = np.zeros((n_sess, n_sess))
for i, s1 in enumerate(session_list):
    for j, s2 in enumerate(session_list):
        sim_sessions[i, j] = np.dot(session_means[s1], session_means[s2])

im = axes[1].imshow(sim_sessions, cmap="YlOrRd", vmin=0.3, vmax=1.0)
axes[1].set_xticks(range(n_sess))
axes[1].set_xticklabels([s.replace("llm_", "") for s in session_list], rotation=45)
axes[1].set_yticks(range(n_sess))
axes[1].set_yticklabels([s.replace("llm_", "") for s in session_list])
axes[1].set_title("Similitud media entre sesiones", fontweight="bold")
fig.colorbar(im, ax=axes[1], shrink=0.8)

for i in range(n_sess):
    for j in range(n_sess):
        color = "white" if sim_sessions[i, j] > 0.7 else "black"
        axes[1].text(j, i, f"{sim_sessions[i,j]:.2f}", ha="center", va="center", fontsize=8, color=color)

# Distribucion de documentos por sesion (donut chart)
session_counts = Counter(session_ids)
labels_d = [s.replace("llm_", "") for s in unique_sessions]
sizes_d = [session_counts[s] for s in unique_sessions]
colors_d = [session_to_color[s] for s in unique_sessions]

wedges, texts, autotexts = axes[2].pie(sizes_d, labels=labels_d, autopct="%1.0f%%",
                                        colors=colors_d, startangle=90,
                                        pctdistance=0.8, textprops={"fontsize": 10})
centre = plt.Circle((0, 0), 0.5, fc="white")
axes[2].add_artist(centre)
axes[2].text(0, 0, f"{sum(sizes_d)}\ndocs", ha="center", va="center", fontsize=14, fontweight="bold")
axes[2].set_title("Documentos por sesion", fontweight="bold")

plt.tight_layout()
plt.show()

---
## 9. Resumen

In [ ]:
print("=" * 70)
print("NOTEBOOK 05: RAG INDEXING COMPLETADO")
print("=" * 70)
print(f"")
print(f"Indices creados:")
print(f"  pseudo_label:  FAISS ({result_enriched.get('count',0)} docs) + BM25 ({len(bm25_enriched.vocab)} terms)")
print(f"  raw_evidence:  FAISS ({result_evidence.get('count',0)} docs) + BM25 ({len(bm25_evidence.vocab)} terms)")
print(f"")
print(f"Modelo: {MODEL_NAME}")
print(f"Dimensiones: {embedding_dim}")
print(f"Index type: IndexFlatIP (cosine similarity)")
print(f"")
print(f"Proximo paso: Notebook 06 (RAG Explorer) o lanzar chatbot con: python src/rag/chat_ui.py")
print("=" * 70)